# Notebook 03 - Exploratory Data Analysis

### HR Analytics: Job Change of Data Scientists

**Objective:** Deep bivariate & multivariate analysis. Go beyond basic charts where discover actual drivers of job-switch intent.
**Input:** `data/processed/hr_cleaned.csv`


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.figsize': (13, 6), 'axes.spines.top': False,
    'axes.spines.right': False, 'axes.titlesize': 13, 'axes.titleweight': 'bold'
})
COLORS = {'primary': '#1B3A6B', 'accent': '#E84855', 'safe': '#2ECC71', 'warn': '#F39C12'}
BINARY_PALETTE = {0: COLORS['safe'], 1: COLORS['accent']}

df = pd.read_csv('../data/processed/hr_cleaned.csv')
print(f'Loaded: {df.shape}  |  Job-change rate: {df["target"].mean()*100:.1f}%')

## 1. Job Change Rate by Key Categorical Features


In [ ]:
def jcr_barplot(ax, col, title, order=None):
    """Plot job-change rate (%) per category with annotation."""
    rates = df.groupby(col)['target'].mean().mul(100).round(1)
    counts = df.groupby(col)['target'].count()
    if order:
        rates  = rates.reindex(order)
        counts = counts.reindex(order)
    colors = [COLORS['accent'] if v > df['target'].mean()*100 else COLORS['primary'] for v in rates.values]
    bars = ax.barh(rates.index.astype(str), rates.values, color=colors, alpha=0.88)
    ax.axvline(df['target'].mean()*100, color=COLORS['warn'], ls='--', lw=1.5, label=f'Overall JCR={df["target"].mean()*100:.1f}%')
    ax.set_xlabel('Job Change Rate (%)')
    ax.set_title(title)
    for bar, (n, v) in zip(bars, zip(counts.values, rates.values)):
        ax.text(v + 0.3, bar.get_y() + bar.get_height()/2, f'{v:.1f}%  (n={n:,})', va='center', fontsize=8)
    ax.legend(fontsize=8)

fig, axes = plt.subplots(2, 3, figsize=(22, 12))
axes = axes.flatten()

exp_order = ['Fresher (<1yr)', 'Junior (1-3yr)', 'Mid (4-7yr)', 'Senior (8-15yr)', 'Veteran (>15yr)']
jcr_barplot(axes[0], 'experience_band', 'JCR by Experience Band', order=exp_order)
jcr_barplot(axes[1], 'city_tier', 'JCR by City Development Tier')
jcr_barplot(axes[2], 'education_level', 'JCR by Education Level',
            order=['Primary School', 'High School', 'Graduate', 'Masters', 'Phd'])
jcr_barplot(axes[3], 'company_size', 'JCR by Company Size')
jcr_barplot(axes[4], 'company_type', 'JCR by Company Type')
jcr_barplot(axes[5], 'enrolled_university', 'JCR by University Enrollment')

plt.suptitle('Job Change Rate (JCR) by Key HR Features', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../data/processed/03_jcr_by_category.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. City Development Index vs Job Change Probability


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Scatter: CDI vs target
for t, color in BINARY_PALETTE.items():
    sub = df[df['target'] == t]
    axes[0].scatter(sub['city_development_index'], sub['training_hours_capped'],
                    alpha=0.15, color=color, s=12, label=['Not Switching', 'Switching'][t])
axes[0].set_xlabel('City Development Index')
axes[0].set_ylabel('Training Hours (Capped)')
axes[0].set_title('CDI vs Training Hours by Job-Switch Intent')
axes[0].legend()

# Binned CDI vs JCR trend
df['cdi_bin'] = pd.cut(df['city_development_index'], bins=10)
cdi_jcr = df.groupby('cdi_bin')['target'].mean().mul(100).reset_index()
cdi_jcr['cdi_mid'] = cdi_jcr['cdi_bin'].apply(lambda x: x.mid)
axes[1].plot(cdi_jcr['cdi_mid'], cdi_jcr['target'], color=COLORS['accent'], lw=2.5, marker='o')
axes[1].fill_between(cdi_jcr['cdi_mid'], cdi_jcr['target'], alpha=0.15, color=COLORS['accent'])
axes[1].set_xlabel('City Development Index')
axes[1].set_ylabel('Job Change Rate (%)')
axes[1].set_title('CDI vs Job Change Rate (Binned Trend)')
axes[1].axhline(df['target'].mean()*100, color=COLORS['warn'], ls='--', label='Overall JCR')
axes[1].legend()

plt.suptitle('City Development Index Analysis', fontweight='bold')
plt.tight_layout()
plt.savefig('../data/processed/03_cdi_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Experience vs Job Change


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# JCR by experience_num (granular)
exp_jcr = df.groupby('experience_num')['target'].mean().mul(100)
exp_cnt = df.groupby('experience_num')['target'].count()
axes[0].bar(exp_jcr.index, exp_jcr.values, color=COLORS['primary'], alpha=0.8, width=0.7)
axes[0].axhline(df['target'].mean()*100, color=COLORS['accent'], ls='--', lw=1.5, label='Overall JCR')
axes[0].set_xlabel('Years of Experience')
axes[0].set_ylabel('Job Change Rate (%)')
axes[0].set_title('Granular: JCR by Exact Experience (Years)')
axes[0].legend()

# Stacked bar by experience band
exp_band_order = ['Fresher (<1yr)', 'Junior (1-3yr)', 'Mid (4-7yr)', 'Senior (8-15yr)', 'Veteran (>15yr)']
exp_ct = df.groupby(['experience_band', 'target']).size().unstack(fill_value=0).reindex(exp_band_order)
exp_pct = exp_ct.div(exp_ct.sum(axis=1), axis=0) * 100
exp_pct[[0, 1]].plot(kind='bar', stacked=True, ax=axes[1],
    color=[COLORS['safe'], COLORS['accent']], edgecolor='white')
axes[1].set_xlabel('Experience Band')
axes[1].set_ylabel('% of Candidates')
axes[1].set_title('Job Switch Composition by Experience Band')
axes[1].legend(['Not Switching', 'Switching'], loc='upper right')
axes[1].tick_params(axis='x', rotation=25)

plt.suptitle('Experience-Based Job Change Analysis', fontweight='bold')
plt.tight_layout()
plt.savefig('../data/processed/03_experience_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Retention Risk Score Distribution


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Risk score distribution by target
for t, color, label in zip([0, 1], [COLORS['safe'], COLORS['accent']], ['Not Switching', 'Switching']):
    axes[0].hist(df[df['target']==t]['retention_risk_score'], bins=30, alpha=0.6, color=color, label=label, density=True)
axes[0].set_xlabel('Retention Risk Score (0–10)')
axes[0].set_ylabel('Density')
axes[0].set_title('Risk Score by Job Switch Intent')
axes[0].legend()

# Risk tier vs JCR
tier_jcr = df.groupby('risk_tier')['target'].mean().mul(100)
tier_cnt = df.groupby('risk_tier')['target'].count()
tier_colors = [COLORS['safe'], COLORS['warn'], COLORS['accent']]
bars = axes[1].bar(tier_jcr.index.astype(str), tier_jcr.values, color=tier_colors, edgecolor='white')
for bar, v, n in zip(bars, tier_jcr.values, tier_cnt.values):
    axes[1].text(bar.get_x()+bar.get_width()/2, v+0.5, f'{v:.1f}%\n(n={n:,})', ha='center', fontsize=9, fontweight='bold')
axes[1].set_title('JCR by Risk Tier')
axes[1].set_ylabel('Job Change Rate (%)')

# Risk tier pie
tier_counts = df['risk_tier'].value_counts().reindex(['Low Risk', 'Medium Risk', 'High Risk'])
axes[2].pie(tier_counts.values, labels=tier_counts.index, colors=tier_colors,
            autopct='%1.1f%%', startangle=90, wedgeprops={'edgecolor':'white','linewidth':2})
axes[2].set_title('Candidate Distribution by Risk Tier')

plt.suptitle('Retention Risk Score Analysis', fontweight='bold')
plt.tight_layout()
plt.savefig('../data/processed/03_risk_score_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Correlation Heatmap


In [ ]:
num_features = [
    'city_development_index', 'training_hours_capped', 'experience_num',
    'education_ordinal', 'last_new_job_num', 'company_size_num',
    'has_relevent_exp', 'is_stem', 'is_enrolled', 'retention_risk_score', 'target'
]
corr_matrix = df[num_features].corr()

fig, ax = plt.subplots(figsize=(13, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
    center=0, vmin=-1, vmax=1, linewidths=0.5, ax=ax,
    annot_kws={'size': 9}
)
ax.set_title('Correlation Heatmap — All Numeric Features vs Target', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('../data/processed/03_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

# Print top correlations with target
print('Top correlations with target (job change):')
target_corr = corr_matrix['target'].drop('target').sort_values(key=abs, ascending=False)
print(target_corr.to_string())

## 6. STEM vs Non-STEM & Gender Analysis


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# STEM vs non-STEM JCR
stem_jcr = df.groupby('major_discipline')['target'].mean().mul(100).sort_values(ascending=True)
colors_stem = [COLORS['accent'] if v > df['target'].mean()*100 else COLORS['primary'] for v in stem_jcr.values]
axes[0].barh(stem_jcr.index, stem_jcr.values, color=colors_stem, alpha=0.88)
axes[0].axvline(df['target'].mean()*100, color=COLORS['warn'], ls='--', label='Overall JCR')
axes[0].set_title('JCR by Major Discipline')
axes[0].set_xlabel('Job Change Rate (%)')
axes[0].legend()

# Gender JCR
gender_jcr = df.groupby('gender')['target'].mean().mul(100)
axes[1].bar(gender_jcr.index, gender_jcr.values, color=COLORS['primary'], alpha=0.85, edgecolor='white')
axes[1].axhline(df['target'].mean()*100, color=COLORS['accent'], ls='--', label='Overall JCR')
for i, (g, v) in enumerate(gender_jcr.items()):
    axes[1].text(i, v+0.5, f'{v:.1f}%', ha='center', fontweight='bold')
axes[1].set_title('JCR by Gender')
axes[1].set_ylabel('Job Change Rate (%)')
axes[1].legend()

# Company type stacked bar
ct = df.groupby(['company_type', 'target']).size().unstack(fill_value=0)
ct_pct = ct.div(ct.sum(axis=1), axis=0) * 100
ct_pct[[0, 1]].plot(kind='bar', stacked=True, ax=axes[2],
    color=[COLORS['safe'], COLORS['accent']], edgecolor='white')
axes[2].set_title('Switch Intent by Company Type')
axes[2].set_ylabel('% of Candidates')
axes[2].legend(['Not Switching', 'Switching'])
axes[2].tick_params(axis='x', rotation=30)

plt.suptitle('Demographics & Background Analysis', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.savefig('../data/processed/03_demographics_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Training Hours vs Job Change


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Box plot
df.boxplot(column='training_hours_capped', by='target', ax=axes[0],
           boxprops=dict(color=COLORS['primary']),
           medianprops=dict(color=COLORS['accent'], linewidth=2))
axes[0].set_title('Training Hours Distribution by Switch Intent')
axes[0].set_xlabel('Target (0=Not Switching, 1=Switching)')
axes[0].set_ylabel('Training Hours')
plt.sca(axes[0])
plt.title('Training Hours vs Job Switch')

# Binned training hours vs JCR
df['training_bin'] = pd.cut(df['training_hours_capped'], bins=8)
train_jcr = df.groupby('training_bin')['target'].mean().mul(100)
axes[1].plot(range(len(train_jcr)), train_jcr.values, color=COLORS['accent'], lw=2, marker='o')
axes[1].set_xticks(range(len(train_jcr)))
axes[1].set_xticklabels([str(b) for b in train_jcr.index], rotation=30, fontsize=8)
axes[1].set_ylabel('Job Change Rate (%)')
axes[1].set_title('JCR by Training Hours Bin')
axes[1].axhline(df['target'].mean()*100, color=COLORS['warn'], ls='--', label='Overall JCR')
axes[1].legend()

plt.suptitle('Training Hours Analysis', fontweight='bold')
plt.tight_layout()
plt.savefig('../data/processed/03_training_hours_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print('Median training hours:')
print(df.groupby('target')['training_hours_capped'].median())

## 8. EDA Summary


In [ ]:
print('='*70)
print('  EDA FINDINGS — Key Drivers of Job Change Intent')
print('='*70)

findings = [
    ('1', 'CDI',        'Candidates from Tier 3 cities (CDI<0.6) show highest JCR — opportunity-seeking'),
    ('2', 'Experience', 'Freshers (<1yr) and Mid-career (4-7yr) have highest JCR — U-shaped pattern'),
    ('3', 'Company',    'Candidates from startups and NGOs switch more than Pvt Ltd employees'),
    ('4', 'Company Sz', 'Unknown company size (likely unemployed) has highest JCR'),
    ('5', 'Education',  'Graduate-level candidates switch more than Masters/PhD holders'),
    ('6', 'Training',   'Lower training hours correlate slightly with higher switch intent'),
    ('7', 'STEM',       'Non-STEM majors show marginally higher JCR than STEM'),
    ('8', 'Gender',     'Male candidates show slightly higher switch rate than female'),
]

for num, feature, insight in findings:
    print(f'  [{num}] {feature:<12}: {insight}')
